# 🔧 02 — Fine-tuning TinyLlama con LoRA
Fine-tuning eficiente usando PEFT + LoRA + TRL.

> **Runtime recomendado:** T4 GPU (gratis) o A100 (Colab Pro)
>
> ⚠️ Antes de correr: `Runtime → Change runtime type → T4 GPU`

## 1. Clonar repo e instalar dependencias

In [ ]:
import os

GITHUB_USER = "Dvilchess"  # <-- cambiá esto
REPO = "lillama"
BRANCH = "dev"

if not os.path.exists(REPO):
    !git clone -b {BRANCH} https://github.com/{GITHUB_USER}/{REPO}
else:
    !cd {REPO} && git pull

%cd {REPO}
!pip install -q -r requirements.txt -r requirements-dev.txt
print('Listo')

## 2. Verificar GPU

In [ ]:
import torch
assert torch.cuda.is_available(), '❌ No hay GPU — cambiá el runtime a T4'
print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 3. Configuración del fine-tuning
Editá estos parámetros según tu dataset y hardware.

In [ ]:
# --- Modelo ---
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./models/finetuned"

# --- Dataset ---
# Opciones: cualquier dataset de HuggingFace con campo 'text' o 'prompt'/'response'
DATASET_NAME = "teknium/OpenHermes-2.5"  # ejemplo, cambiá por el tuyo
DATASET_SPLIT = "train[:2000]"           # primeros 2000 ejemplos para prueba

# --- LoRA ---
LORA_R = 16           # rank — más alto = más parámetros entrenables
LORA_ALPHA = 32       # escala
LORA_DROPOUT = 0.05

# --- Entrenamiento ---
EPOCHS = 1
BATCH_SIZE = 4
GRAD_ACCUM = 4        # batch efectivo = BATCH_SIZE * GRAD_ACCUM = 16
LR = 2e-4
MAX_SEQ_LEN = 512

print('Config lista')

## 4. Cargar modelo en 4-bit (QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Cargar y preparar dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
print(f'Ejemplos: {len(dataset)}')
print(f'Columnas: {dataset.column_names}')
print('\nEjemplo:')
print(dataset[0])

In [ ]:
# Formateador al chat template de TinyLlama
def format_example(example):
    # Ajustá según las columnas de tu dataset
    if 'conversations' in example:
        # formato OpenHermes
        msgs = [{"role": "user" if m['from'] == 'human' else "assistant", "content": m['value']}
                for m in example['conversations']]
    elif 'prompt' in example and 'response' in example:
        msgs = [
            {"role": "user", "content": example['prompt']},
            {"role": "assistant", "content": example['response']},
        ]
    else:
        msgs = [{"role": "user", "content": example.get('text', '')}]

    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}

dataset = dataset.map(format_example)
print('Dataset formateado')
print(dataset[0]['text'][:300])

## 6. Fine-tuning con SFTTrainer

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    max_seq_length=MAX_SEQ_LEN,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

trainer.train()

## 7. Guardar modelo fine-tuneado

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Modelo guardado en {OUTPUT_DIR}')

## 8. Subir modelo a HuggingFace Hub (opcional)
Para usarlo desde prod sin tener que re-entrenar.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

# HF_REPO = "TU_USUARIO/tinyllama-finetuned"  # <-- cambiá esto
# model.push_to_hub(HF_REPO)
# tokenizer.push_to_hub(HF_REPO)
# print(f'Subido a https://huggingface.co/{HF_REPO}')
print('Descomentá las líneas de arriba cuando quieras subir a HF Hub')

## 9. Prueba rápida del modelo fine-tuneado

In [ ]:
import sys
sys.path.insert(0, '.')
os.environ['ENV'] = 'prod'
os.environ['FINETUNED_MODEL_DIR'] = OUTPUT_DIR

from app.model import LLMModel
finetuned = LLMModel()
finetuned.load()

print(finetuned.generate('¿Cuál es la capital de Francia?', 'Eres un asistente útil.'))

## 10. Commit del modelo al repo (opcional)

In [ ]:
# ⚠️ Los modelos son grandes — mejor subirlos a HF Hub (paso 8)
# Solo hacer esto si el modelo es pequeño o usás Git LFS

# !git config user.email "tu@email.com"
# !git config user.name "Tu Nombre"
# !git add models/finetuned/
# !git commit -m 'feat: add finetuned model'
# !git push origin dev
print('Ver comentarios arriba antes de hacer push del modelo')